# Phase 3: Reliability Analysis

Question: Are neurons with sharper tuning also more reliable?


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
FINAL = os.path.join(BASE, 'data', 'final')

df = pd.read_parquet(os.path.join(FINAL, 'orientation_neuron_analysis.parquet'))
df_valid = df[df['fit_success'] & df['split_half_reliability'].notna()]
kappa = df_valid['fit_kappa_or_width'].values
reliability = df_valid['split_half_reliability'].values
mean_resp = df_valid['mean_response'].values
print(f'n = {len(df_valid)}')


## OLS Model: reliability ~ kappa + mean_response

In [ ]:
n = len(df_valid)
X = np.column_stack([np.ones(n), kappa, mean_resp])
beta, _, _, _ = np.linalg.lstsq(X, reliability, rcond=None)
y_pred = X @ beta
resid = reliability - y_pred
sigma2 = np.sum(resid**2) / (n - 3)
se = np.sqrt(sigma2 * np.diag(np.linalg.inv(X.T @ X)))
t_stat = beta / se
p_val = 2 * (1 - sp_stats.t.cdf(np.abs(t_stat), df=n-3))
r2 = 1 - np.sum(resid**2) / np.sum((reliability - reliability.mean())**2)

print(f'R2 = {r2:.4f}')
for name, b, s, t, p in zip(['intercept', 'kappa', 'mean_response'], beta, se, t_stat, p_val):
    print(f'  {name}: beta={b:.6f}, SE={s:.6f}, t={t:.2f}, p={p:.2e}')


## Correlations

In [ ]:
r_p, p_p = sp_stats.pearsonr(kappa, reliability)
r_s, p_s = sp_stats.spearmanr(kappa, reliability)
print(f'Pearson:  r={r_p:.4f}, p={p_p:.2e}')
print(f'Spearman: r={r_s:.4f}, p={p_s:.2e}')


## Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(kappa, reliability, s=1, alpha=0.15, color='steelblue')
k_range = np.linspace(0, 20, 200)
ax.plot(k_range, beta[0] + beta[1]*k_range + beta[2]*mean_resp.mean(),
        '-', color='red', lw=2, label=f'OLS (beta_k={beta[1]:.4f})')
ax.set_xlabel('Kappa'); ax.set_ylabel('Reliability')
ax.set_title('Reliability vs Tuning Sharpness')
ax.legend(); plt.tight_layout(); plt.show()


## Model Results Table

In [ ]:
results = pd.read_csv(os.path.join(BASE, 'reports', 'tables', 'reliability_model_results.csv'))
results
